In [1]:
import pandas as pd
import stumpy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as dates
from matplotlib.patches import Rectangle
import datetime as dt


plt.style.use('https://raw.githubusercontent.com/TDAmeritrade/stumpy/main/docs/stumpy.mplstyle')

In [ ]:
steam_df = pd.read_csv("https://zenodo.org/record/4273921/files/STUMPY_Basics_steamgen.csv?download=1")
steam_df.head()

In [ ]:
plt.suptitle('Steamgen Dataset', fontsize='30')
plt.xlabel('Time', fontsize ='20')
plt.ylabel('Steam Flow', fontsize='20')
plt.plot(steam_df['steam flow'].values)
plt.show()

In [ ]:
m = 640
fig, axs = plt.subplots(2)
plt.suptitle('Steamgen Dataset', fontsize='30')
axs[0].set_ylabel("Steam Flow", fontsize='20')
axs[0].plot(steam_df['steam flow'], alpha=0.5, linewidth=1)
axs[0].plot(steam_df['steam flow'].iloc[643:643+m])
axs[0].plot(steam_df['steam flow'].iloc[8724:8724+m])
rect = Rectangle((643, 0), m, 40, facecolor='lightgrey')
axs[0].add_patch(rect)
rect = Rectangle((8724, 0), m, 40, facecolor='lightgrey')
axs[0].add_patch(rect)
axs[1].set_xlabel("Time", fontsize='20')
axs[1].set_ylabel("Steam Flow", fontsize='20')
axs[1].plot(steam_df['steam flow'].values[643:643+m], color='C1')
axs[1].plot(steam_df['steam flow'].values[8724:8724+m], color='C2')
plt.show()

In [ ]:
m = 640
mp = stumpy.stump(steam_df['steam flow'], m)

plt.plot(mp[:,0])

In [ ]:
motif_idx = np.argsort(mp[:, 0])[0]

print(f"The motif is located at index {motif_idx}")

In [ ]:
nearest_neighbor_idx = mp[motif_idx, 1]

print(f"The nearest neighbor is located at index {nearest_neighbor_idx}")

In [ ]:
discord_idx = np.argsort(mp[:, 0])[-1]

print(f"The discord is located at index {discord_idx}")

In [ ]:
nearest_neighbor_distance = mp[discord_idx, 0]

print(f"The nearest neighbor subsequence to this discord is {nearest_neighbor_distance} units away")

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0})
plt.suptitle('Discord (Anomaly/Novelty) Discovery', fontsize='30')

axs[0].plot(steam_df['steam flow'].values)
axs[0].set_ylabel('Steam Flow', fontsize='20')
rect = Rectangle((discord_idx, 0), m, 40, facecolor='lightgrey')
axs[0].add_patch(rect)
axs[1].set_xlabel('Time', fontsize ='20')
axs[1].set_ylabel('Matrix Profile', fontsize='20')
axs[1].axvline(x=discord_idx, linestyle="dashed")
axs[1].plot(mp[:, 0])
plt.show()

In [11]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import stumpy


plt.style.use('https://raw.githubusercontent.com/TDAmeritrade/stumpy/main/docs/stumpy.mplstyle')

In [ ]:
df = pd.read_csv('https://zenodo.org/record/5045218/files/hemoglobin.csv?download=1')
df = df[6000:14000]
df = df.reset_index(drop=True)
df.head()

In [ ]:
plt.plot(df['Hemoglobin Concentration'])
plt.xlabel('Time', fontsize="20")
plt.ylabel('Intensity', fontsize="20")
plt.title('Hemoglobin Concentration', fontsize="30")
plt.show()

In [29]:
m = 600
mp = stumpy.stump(df['Hemoglobin Concentration'], m)
motif_idx = np.argmin(mp[:, 0])
nn_idx = mp[motif_idx, 1]

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0})
plt.suptitle('Motif (Pattern) Discovery', fontsize='30')

axs[0].plot(df['Hemoglobin Concentration'])
axs[0].set_ylabel('Intensity', fontsize='20')
rect = Rectangle((motif_idx, 4000), m, 4000, facecolor='lightgrey')
axs[0].add_patch(rect)
rect = Rectangle((nn_idx, 4000), m, 4000, facecolor='lightgrey')
axs[0].add_patch(rect)
axs[1].set_xlabel('Time', fontsize ='20')
axs[1].set_ylabel('Matrix Profile', fontsize='20')
axs[1].axvline(x=motif_idx, linestyle="dashed")
axs[1].axvline(x=nn_idx, linestyle="dashed")
axs[1].plot(mp[:, 0])
plt.show()

plt.show()

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0})

axs[0].plot(df['Hemoglobin Concentration'])
axs[0].set_ylabel('Intensity', fontsize="20")
axs[0].set_title('Hemoglobin Concentration', fontsize="30")

axs[1].plot(df['Sensor Acceleration'])
axs[1].set_ylabel("Sensor\nAcceleration", fontsize="20")
axs[1].set_xlabel("Time", fontsize="20")
rect = Rectangle((3500, 6200), 3750, 3000, facecolor='lightgrey')
axs[1].add_patch(rect)

plt.show()

In [32]:
variance = pd.Series(df['Sensor Acceleration']).rolling(m).var().dropna().values
annotation_vector = (variance < 10000).astype(np.float64)

In [ ]:
fig, axs = plt.subplots(3, sharex=True, gridspec_kw={'hspace': 0})
plt.suptitle("Annotation Vector Generation", fontsize="30")

axs[0].plot(df["Sensor Acceleration"])
axs[0].set_ylabel("Sensor\nAcceleration", fontsize="20")
axs[1].plot(variance)
axs[1].set_ylabel("Variance", fontsize="20")
axs[2].plot(annotation_vector)
axs[2].set_ylabel("Annotation\nValue", fontsize="20")
axs[2].set_xlabel('Time', fontsize="20")

plt.show()

In [ ]:
corrected_mp = mp[:, 0] + ((1 - annotation_vector) * np.max(mp[:, 0]))

fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0})
plt.suptitle('Matrix Profile Comparison', fontsize="30")

axs[0].plot(mp[:, 0], linestyle='--')
axs[0].plot(corrected_mp, color='C1', linewidth=3)
axs[0].set_ylabel("(Corrected)\nMatrix Profile", fontsize="20")

axs[1].plot(annotation_vector)
axs[1].set_xlabel("Time", fontsize="20")
axs[1].set_ylabel("Annotation\nValue", fontsize="20")

plt.show()

In [37]:
motif_idx = np.argmin(corrected_mp)
nn_idx = mp[motif_idx, 1]

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0})
plt.suptitle('Motif (Pattern) Discovery', fontsize='30')

axs[0].plot(df['Hemoglobin Concentration'])
axs[0].set_ylabel('Intensity', fontsize='20')
rect = Rectangle((motif_idx, 4000), m, 4000, facecolor='lightgrey')
axs[0].add_patch(rect)
rect = Rectangle((nn_idx, 4000), m, 4000, facecolor='lightgrey')
axs[0].add_patch(rect)
axs[1].set_xlabel('Time', fontsize ='20')
axs[1].set_ylabel('Corrected\nMatrix Profile', fontsize='20')
axs[1].axvline(x=motif_idx, linestyle="dashed")
axs[1].axvline(x=nn_idx, linestyle="dashed")
axs[1].plot(corrected_mp)
plt.show()

plt.show()

## Anomaly detection in multidimensional time series

In [2]:
import numpy as np
import matplotlib as mplt
from matplotlib import pyplot as plt
from statsmodels.tsa.tsatools import detrend

import pandas as pd
import stumpy

In [ ]:
# Generating time series

C = 5
N = 300

sig = np.array([1, 2, 1, 2, 1])*0.01
xo = np.arange(N)

t0 = np.zeros((C,N),dtype=float)
r = np.zeros((C,N),dtype=float)

for i in range(C):
    r[i,:] += np.random.normal(0,sig[i],N)
    r[i,:] += 0.3*np.sin(2*xo) + 0.2*np.sin(5*xo) + 0.8*np.sin(1*xo) + 0.2*np.sin(9*xo)
    #r[i,:] += 0*np.sin(0.01*xo)


s = np.sin(0.2*xo)
#s[32:48] *= 2
#s[221:235] *= 2



fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1.5*C])
if C == 1:
    axs = [axs]
for i in range(C):
    t0[i,:] = s + r[i,:]
    axs[i].plot(xo,t0[i,:]);
    axs[i].set_ylabel(f'Sinal {i+1}')
    axs[i].grid(True)

axs[i].set_xlabel('Amostra');




In [ ]:
# Inserting anomalies

# size = N//10
# anomalies = [(2,int(0.2*N),size,[]), (3,int(0.5*N),size,[]),(1,int(0.8*N),size,[])]
# to = t0.copy()

fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1*C])
# if C == 1:
#     axs = [axs]
# for o in range(len(anomalies)):
#     n,i,m,_  = anomalies[o]
#     series = np.random.default_rng().choice(C,size=n,replace=False)
#     anomalies[o][3].extend(series)
#     print(series, i)

#     for s in series:
#         to[s,i:i+m] += 2*np.random.rand(m)

#         #axs[s].axvline(x=i, color='k', ls='--', lw=2)
#         #axs[s].axvline(x=i+m, color='r', ls='--', lw=2)


for i in range(C):
    axs[i].plot(xo,to[i,:]);
    axs[i].set_ylim([-2.5,2.5])
    axs[i].set_ylabel(f'Signal {i+1}')
    axs[i].grid(True)

axs[i].set_xlabel('Timestamp')

In [ ]:
# Smoothing

m = 10
t = np.empty_like(to)
fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,2*C])
if C == 1:
    axs = [axs]
for i in range(C):
    t[i,:] = pd.Series(to[i,:]).rolling(m).mean().values
    axs[i].plot(t[i,:]);
    axs[i].set_ylabel(f'Sinal {i+1}')
    axs[i].grid(True)

axs[i].set_xlabel('Amostra')

In [ ]:
from scipy.signal import butter, lfilter, freqz
# Create filter

fs = 30
cutoff = 3
order = 12

b,a = butter(order, cutoff,fs=fs,btype='low',analog=False)

w, h = freqz(b, a, fs=fs)

plt.subplot(2, 1, 1)
plt.plot(w, np.abs(h), 'b')
plt.plot(cutoff, 0.5*np.sqrt(2), 'ko')
plt.axvline(cutoff, color='k')
plt.xlim(0, 0.5*fs)
plt.title("Lowpass Filter Frequency Response")
plt.xlabel('Frequency [Hz]')
plt.grid()


In [ ]:
from statsmodels.tsa.filters.cf_filter import cffilter

# Filter
on = True

C = 5
fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1*C])
if C == 1:
    axs = [axs]
if on:
    t = np.empty_like(to)
    x = xo.copy()

    for i in range(C):
        t[i,:] = lfilter(b,a,to[i,:])
        axs[i].plot(xo,t[i,:]);
        axs[i].set_ylabel(f'Filtered {i+1}')
        axs[i].set_ylim([-3,3])
        axs[i].grid(True)

    axs[i].set_xlabel('Timestamp')
    
else:
    t = to.copy()
    x = xo.copy()

    for i in range(C):
        axs[i].plot(x,t[i,:]);
        axs[i].set_ylabel(f'Sinal {i+1}')
        axs[i].set_ylim([-2.5,2.5])
        axs[i].grid(True)
        

axs[i].set_xlabel('Timestamp')

In [ ]:
# Getting matrix profiles
m = 40
MPs = np.zeros((C,x.size-m+1),dtype=float)


for k in range(C):
    mp = stumpy.stump(t[k,:],m)[:,0]
    mp -= np.quantile(mp,.75)
    mp[mp<0] = 0
    mp[:10] = 0
    MPs[k,:] = mp
    
    
KDP = np.sort(MPs,axis=0)

# First plot
max_y = np.max(MPs)
fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1.25*C])
if C == 1:
    axs = [axs]
for k in range(C):
    axs[k].plot(MPs[k,:]);
    axs[k].set_ylim([0,max_y])
    axs[k].grid(True)
    axs[k].set_ylabel(f'MP {k+1}')
    axs[k].set_xlim([0,300])
axs[k].set_xlabel('Timestamp')

for n,i,_,series in anomalies:
    for o in series:
        rect = Rectangle((i-m//2, 0), m+20, max_y, facecolor='lightgrey')
        axs[o].add_patch(rect)


# Second plot
fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1.25*C])
if C == 1:
    axs = [axs]
for k in range(C):
    for n,i,_,_ in anomalies:
        if C-k <= n:
            rect = Rectangle((i-m//2, 0), m+20, max_y, facecolor='lightgrey')
            axs[C-k-1].add_patch(rect)
    axs[C-k-1].plot(KDP[k,:]);
    axs[C-k-1].set_ylim([0,max_y])
    axs[C-k-1].set_xlim([0,300])
    axs[C-k-1].grid(True)

    axs[C-k-1].set_ylabel(f'{C-k}-KDP')

axs[k].set_xlabel('Timestamp')

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import animation
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 150  
plt.ioff()

# Getting matrix profiles
m = 30
mp = stumpy.stump(t[0,:],m)


l = 4 + 38



loc = mp[:,1]
mp = mp[:,0]
#mp -= np.quantile(mp,.75)
#mp[mp<0] = 0


# First plot
max_y = np.max(mp)

fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,4])


axs[0].plot(xo,to[0,:]);
axs[0].set_ylabel(f'Sinal')
axs[0].grid(True)

i = 90
rect1 = Rectangle((i, -1.5), m, 3.5, facecolor='lightblue')
axs[0].add_patch(rect1)

j = 50
rect2 = Rectangle((j, -1.5), m, 3.5, facecolor='orange')
axs[0].add_patch(rect2)

recs = [rect1, rect2]

axs[1].plot(np.arange(15,N-14),mp);
axs[1].plot([90+15],[mp[90]],'ro')
axs[1].set_xlim([0,300]);
axs[1].set_ylim([0.75,2.5]);
axs[1].set_xlabel('Amostra')
axs[1].set_ylabel(f'Matrix Profile')
axs[1].grid(True)

def animate(i):
    
    if i >= 13:
        i += 11

    if i<l+13:
        i *= 5
        recs[1].set_xy([i,-1.5])
    else:
         recs[1].set_xy([loc[90],-1.5])

    return recs,

ani = animation.FuncAnimation(fig, animate, frames=l+50,interval=150)


ani.save('./anim.gif', writer='imagemagick', fps=15)



# Filtering

In [ ]:
from scipy.signal import butter, lfilter, freqz
# Create filter

fs = 24
cutoff = 4
order = 10

num_butter,den_butter = butter(order, cutoff,fs=fs,btype='low')

w, h = freqz(num_butter, den_butter, fs=fs)

plt.subplot(2, 1, 1)
plt.plot(w, np.abs(h), 'b')
plt.plot(cutoff, 0.5*np.sqrt(2), 'ko')
plt.axvline(cutoff, color='k')
plt.xlim(0, 0.5*fs)
plt.title("Lowpass Filter Frequency Response")
plt.xlabel('Frequency [Hz]')
plt.grid()



In [ ]:
N = 30
n = 5
den_diff = np.array([1])
num_diff = np.concatenate(([n-1],[-1]*(n-1)))

den = np.polymul(den_butter,den_diff)
num = np.polymul(num_butter,num_diff)

w, h = freqz(num, den, fs=N)

plt.subplot(2, 1, 1)
plt.plot(w, np.abs(h), 'b')
plt.title("Lowpass Filter Frequency Response")
plt.xlabel('Frequency [Hz]')
plt.grid()


In [ ]:
from importlib import reload
import ESP_funcs as esp
# Getting part of data

year = 2014
month1 = 5
month2 = 9
well = 'A-24 1'
data_name = 'ESP motor temperature'


reload(esp)
data_full = esp.load_data()
blcks = esp.get_blocks(data_full, well)

well_data = data_full[data_full['Well Run'] ==well]
ind = blcks[1]


#ind = ind[np.logical_and(ind.year == year,np.logical_and(ind.month>=month1,ind.month<month2))]
y = well_data.loc[:,data_name]

yorig = y.copy()


y1, outliers = esp.remove_outliers(y, nIQR = 1,diff=1)

y2, outliers = esp.remove_outliers(y1, nIQR = 1.5,diff=0)

ind = y.index

plt.figure(0)
yorig.plot(style='-')

plt.ylabel('ESP discharge pressure [bar]')
plt.grid(True)
plt.gcf().set_size_inches(12,5)

plt.figure(1)
y1.plot(style='-')

plt.ylabel('ESP discharge pressure [bar]')
plt.grid(True)
plt.gcf().set_size_inches(12,5)

plt.figure(2)
y2.plot(style='-')

plt.ylabel('ESP discharge pressure [bar]')
plt.grid(True)
plt.gcf().set_size_inches(12,5)


In [ ]:
reload(esp)
yor = y2.copy()
yfill, intervs = esp.fill_gaps(yor,grace_after=0,min_size=24)


yor.plot()
yfill.plot()
esp.plot_recs(intervs,plt.gca(),yfill)



In [ ]:

fs = 1
T_cutoff = 24
cutoff = 1/T_cutoff
btype = 'low'
order = 6
n = 2

den_diff = 1
num_diff = 1
num_butter = 1
den_butter = 1

use_butter = 1
use_mean = 1

if use_mean:
    den_diff = np.array([1])
    num_diff = np.concatenate(([n-1],[-1]*(n-1)))/n
    #num_diff = np.array(([1]*n))/n


if use_butter:
    num_butter,den_butter = butter(order, cutoff,fs=fs,btype=btype)

den = np.polymul(den_butter,den_diff)
num = np.polymul(num_butter,num_diff)

w, h = freqz(num, den, fs=fs)

plt.subplot(2, 1, 1)
plt.plot(w, np.abs(h), 'b')
plt.grid(True)
#plt.subplot(2, 1, 2)
#plt.plot(w, (np.angle(h)*180/np.pi), 'b')
plt.xlabel('Frequency [1/h]')
plt.grid(True)


In [ ]:

#tor = lfilter(num,den,yor-yor[0])
#tgap = lfilter(num,den,ygap-ygap[0])
tfill = lfilter(num,den,yfill-yfill[0])

#plt.plot(yor.index,tor,'b');
#plt.plot(ygap.index,tgap,'r');
plt.plot(yfill.index,tfill,'g');

esp.plot_recs(intervs,plt.gca(),tfill)


In [ ]:
m = 24*1
mp = stumpy.stump(tfill,m)[:,0]
mp -= np.quantile(mp,.75)
mp[mp<0] = 0

plt.plot(yfill.index[:-m+1], mp)
esp.plot_recs(intervs,plt.gca(),mp)


# Filter all data

In [157]:
from importlib import reload
import ESP_funcs as esp
reload(esp)
data_full = esp.load_data()

In [158]:
def load_well(well,data_full):

    if well not in data_full['Well Run'].unique():
        print('Well does not exist')
        return None

    well_data = data_full[data_full['Well Run'] ==well]

    if pd.isna(well_data['Failure distance'].iloc[0]):
        return None

    all_data = well_data.select_dtypes(include=[float])
    all_data.drop(list(all_data.filter(regex='outlier_')),inplace=True,axis=1)
    all_data.drop(list(all_data.filter(regex='extValues_')),inplace=True,axis=1)
    all_data.drop(list(all_data.filter(regex='peaks_')),inplace=True,axis=1)
    all_data.drop(list(all_data.filter(regex='Well aligned')),inplace=True,axis=1)
    all_data.drop(list(all_data.filter(regex='Choke')),inplace=True,axis=1)

    data_out = pd.DataFrame(columns=all_data.columns)

    #data_out.replace(0,np.nan,inplace=True)
    #data_out[data_out<1e-1] = np.nan

    for data_name in all_data.columns:

        data_out[data_name] = well_data.loc[:,data_name]
        #y = well_data.loc[:,data_name]

        #y1, outliers = esp.remove_outliers(y, nIQR = 1,diff=1)

        #data_out[data_name], outliers = esp.remove_outliers(y1, nIQR = 1.5,diff=0)

        if data_out[data_name].isnull().all():
            data_out.drop(data_name,axis=1,inplace=True)

        
        elif len(np.unique(data_out[data_name].dropna()))==1:
            data_out.drop(data_name,axis=1,inplace=True)

        data_out.replace(0,np.nan,inplace=True)
    return data_out

#print(data_out.columns)

In [ ]:
from scipy.signal import butter, lfilter, freqz
from scipy.linalg import pascal
# Set filter
fs = 1
btype = 'pass'
if btype == 'pass':
    T_cutoff = np.array([50,6])
else:
    T_cutoff = 6

cutoff = 1/T_cutoff
order = 7




den_diff = 1
num_diff = 1
num_butter = 1
den_butter = 1

use_butter = 1
use_mean = 0

n = 20
if use_mean:
    #den_diff = np.array([1])
    #num_diff = np.concatenate(([n-1],[-1]*(n-1)))/n
    #num_diff = np.array(([1]*n))/n
    num_diff = pascal(n+1,kind='lower')[n,:n+2] * (-1)**np.arange(n+1) / 2**n
    #num_diff = np.array([1,-1])

if use_butter:
    num_butter,den_butter = butter(order, cutoff,fs=fs,btype=btype)

den = np.polymul(den_butter,den_diff)
num = np.polymul(num_butter,num_diff)

w, h = freqz(num, den, fs=fs)

plt.subplot(2, 1, 1)
plt.loglog(w, np.abs(h), 'b')
plt.grid(True)
plt.subplot(2, 1, 2)
plt.plot(w, np.abs(h), 'b')
#plt.plot(w, (np.angle(h)*180/np.pi), 'b')
plt.xlabel('Frequency [1/h]')
plt.grid(True)




In [ ]:
import datetime
import os

wells = np.unique(data_full['Well Run'])
#wells = ['A-06 2']

dir_name = './Figures'
reload(esp)
#data_full = esp.load_data()
if not os.path.isdir(dir_name):
    os.mkdir(dir_name)

for well in wells:

    try:

        well_dir = f'{dir_name}/{well}'

        if not os.path.isdir(well_dir):
            os.mkdir(well_dir)
        
        data_out = load_well(well,data_full)

        if data_out is None:
            continue

        
        columns = data_out.columns
        C = columns.size
        N = int((data_out.index.max()-data_out.index.min()).total_seconds())//3600 + 1


        plot_mp = 1
        calc_mp = 1

        if not calc_mp: plot_mp = False

        fig1, axs1 = plt.subplots(2*C, sharex=True, gridspec_kw={'hspace': 0},figsize=(16,2*C*2))
        if plot_mp:
            fig2, axs2 = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=(16,2*C))
        if calc_mp:
            fig3, axs3 = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=(16,2*C))

        # Getting matrix profiles
        m = 24*1
        MPs = np.zeros((C,N-m+1),dtype=float)


        for i, data_name in enumerate(columns):

            # Filtering and plotting filtered data
            yor = data_out[data_name]
            yfill, intervs = esp.fill_gaps(yor,grace_after=0,min_size=24*1)

            tfill = pd.Series(data=lfilter(num,den,yfill-yfill[0]),index=yfill.index)

            tfill /= tfill.dropna().std()

            axs1[2*i].plot(yfill.index,yfill,'b');
            axs1[2*i+1].plot(yfill.index,tfill,'b');
            esp.plot_recs(intervs,axs1[2*i],yfill)
            esp.plot_recs(intervs,axs1[2*i+1],tfill)
            axs1[2*i].set_ylabel(data_name)

            # Calculating matrix profile
            if calc_mp:
                mp = esp.get_matrix_profile(tfill,m,intervals=intervs,quartile=0.75)

                MPs[i,:] = mp

                dates = yfill.index[:-m+1]
                if plot_mp:
                    axs2[i].plot(dates,mp);
                    axs2[i].set_ylabel(data_name)
                    esp.plot_recs(intervs,axs2[i],data=mp,add_pre=m)

        fig1.savefig(f'{well_dir}/data.png')
        if plot_mp:
            fig2.savefig(f'{well_dir}/mps.png')

        if calc_mp:

            # Plot sorted profile
            KDP = np.sort(MPs,axis=0)
            max_y = np.nanmax(MPs[~np.isinf(MPs)])
            for k, data_name in enumerate(columns):

                axs3[k].plot(dates,KDP[-k-1,:]);
        #     axs3[k].set_ylim([0,max_y])
                axs3[k].set_ylabel(f'{k+1}-KDP')
            
            fig3.savefig(f'{well_dir}/sorted.png')

    except:
        continue